In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
import time
import pickle
from copy import deepcopy
from collections import deque
import os

# --------------------------------------------------
#  CONFIGURATION (easy to tweak)
# --------------------------------------------------
ROWS, COLS, INAROW = 9, 10, 5           # board dimensions
NUM_ITERATIONS  = 50                     # how many self-play → train cycles
GAMES_PER_ITER  = 40                     # games generated per iteration
MCTS_SIMULATIONS = 400                   # simulations per move during self‑play
EVAL_SIMULATIONS = 800                   # simulations during evaluation matches
BATCH_SIZE       = 128                   # training batch size
EPOCHS_PER_ITER  = 10                    # training epochs per iteration
LR               = 1e-3                  # learning rate
WEIGHT_DECAY     = 1e-4
BUFFER_CAPACITY  = 200_000               # max positions in replay buffer
MIN_BUFFER_SIZE  = 5_000                 # don't train until we have this many
NUM_FILTERS      = 64                    # conv filters in the network
C_PUCT           = 2.0                   # PUCT exploration constant
DIRICHLET_ALPHA  = 0.3                   # Dirichlet noise parameter
DIRICHLET_EPS    = 0.25                  # weight of Dirichlet noise
TEMPERATURE_THRESH = 15                  # first N moves use temp=1, then greedy

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Paths
MODEL_DIR = Path("../ml/models")
DATA_DIR  = Path("../data")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / "selfplay").mkdir(parents=True, exist_ok=True)

In [ ]:
class ConnectFour:
    def __init__(self, rows=ROWS, cols=COLS, inarow=INAROW):
        self.rows = rows
        self.cols = cols
        self.inarow = inarow
        self.board = np.zeros((rows, cols), dtype=np.int8)
        self.current_player = 0
        self.done = False
        self.winner = None

    def clone(self):
        new = ConnectFour(self.rows, self.cols, self.inarow)
        new.board = self.board.copy()
        new.current_player = self.current_player
        new.done = self.done
        new.winner = self.winner
        return new

    def legal_moves(self):
        if self.done:
            return []
        return [c for c in range(self.cols) if self.board[0, c] == 0]

    def apply_move(self, col):
        if col not in self.legal_moves():
            return False
        for r in range(self.rows - 1, -1, -1):
            if self.board[r, col] == 0:
                self.board[r, col] = self.current_player + 1
                break
        self._check_termination(r, col)
        self.current_player = 1 - self.current_player
        return True

    def _check_termination(self, row, col):
        player = self.board[row, col]
        directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
        for dr, dc in directions:
            count = 1
            r, c = row + dr, col + dc
            while 0 <= r < self.rows and 0 <= c < self.cols and self.board[r, c] == player:
                count += 1
                r += dr
                c += dc
            r, c = row - dr, col - dc
            while 0 <= r < self.rows and 0 <= c < self.cols and self.board[r, c] == player:
                count += 1
                r -= dr
                c -= dc
            if count >= self.inarow:
                self.done = True
                self.winner = self.current_player  # player who just moved
                return
        if len(self.legal_moves()) == 0:
            self.done = True
            self.winner = None  # draw

    def display(self):
        symbols = {0: '.', 1: 'X', 2: 'O'}
        for r in range(self.rows):
            print(' '.join(symbols[self.board[r, c]] for c in range(self.cols)))
        print()

In [ ]:
class ConnectFiveNet(nn.Module):
    """One network, two heads: policy (which column) and value (win probability)."""
    def __init__(self, rows=ROWS, cols=COLS, num_filters=NUM_FILTERS):
        super().__init__()
        # Shared convolutional body
        self.conv1 = nn.Conv2d(2, num_filters, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(num_filters)
        self.conv2 = nn.Conv2d(num_filters, num_filters, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(num_filters)
        self.conv3 = nn.Conv2d(num_filters, num_filters, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm2d(num_filters)

        # Policy head
        self.policy_conv = nn.Conv2d(num_filters, 32, kernel_size=1)
        self.policy_bn   = nn.BatchNorm2d(32)
        self.policy_fc   = nn.Linear(32 * rows * cols, cols)

        # Value head
        self.value_conv = nn.Conv2d(num_filters, 32, kernel_size=1)
        self.value_bn   = nn.BatchNorm2d(32)
        self.value_fc1  = nn.Linear(32 * rows * cols, 64)
        self.value_fc2  = nn.Linear(64, 1)

    def forward(self, x):
        # Shared body
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))

        # Policy head
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.reshape(p.size(0), -1)
        p_logits = self.policy_fc(p)

        # Value head
        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.reshape(v.size(0), -1)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v))          # range [-1, 1]

        return p_logits, v.squeeze(-1)


# ---------- board → tensor helper ----------
def board_to_tensor(board_2d: np.ndarray, current_player: int) -> np.ndarray:
    """
    Convert (9,10) board (0=empty, 1=p0, 2=p1) into (2,9,10) from `current_player`'s view.
    Channel 0 = current player's pieces, Channel 1 = opponent's pieces.
    """
    tensor = np.zeros((2, ROWS, COLS), dtype=np.float32)
    tensor[0] = (board_2d == current_player + 1).astype(np.float32)
    tensor[1] = (np.logical_and(board_2d != 0, board_2d != current_player + 1)).astype(np.float32)
    return tensor


# ---------- inference helper (returns numpy) ----------
def network_eval(model: ConnectFiveNet, state: ConnectFour) -> tuple:
    """Run the network on `state` and return (policy_logits, value) as numpy arrays."""
    tensor = board_to_tensor(state.board, state.current_player)
    x = torch.from_numpy(tensor).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits, value = model(x)
    return logits.cpu().numpy().flatten(), value.cpu().item()

In [ ]:
class ReplayBuffer:
    """Stores (state_tensor, policy_target, value_target) tuples."""
    def __init__(self, capacity=BUFFER_CAPACITY):
        self.buffer = deque(maxlen=capacity)

    def add(self, state_tensor, policy_target, value_target):
        self.buffer.append((state_tensor, policy_target, value_target))

    def extend(self, positions):
        """`positions` is a list of (state_tensor, policy_target, value_target)."""
        for pos in positions:
            self.add(*pos)

    def sample(self, batch_size):
        idx = np.random.choice(len(self.buffer), size=batch_size, replace=False)
        states  = np.stack([self.buffer[i][0] for i in idx])
        policies = np.stack([self.buffer[i][1] for i in idx])
        values   = np.array([self.buffer[i][2] for i in idx], dtype=np.float32)
        return (torch.from_numpy(states).to(DEVICE),
                torch.from_numpy(policies).to(DEVICE),
                torch.from_numpy(values).to(DEVICE))

    def __len__(self):
        return len(self.buffer)

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(list(self.buffer), f)

    def load(self, path):
        with open(path, "rb") as f:
            self.buffer = deque(pickle.load(f), maxlen=BUFFER_CAPACITY)

In [ ]:
class MCTSNode:
    def __init__(self, state: ConnectFour, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action          # column that led here from parent
        self.children = {}            # col → MCTSNode
        self.visit_count = 0
        self.total_value = 0.0        # sum of values from this node's perspective (player to move)
        self.prior = {}               # col → prior probability (from network)
        self.untried_actions = state.legal_moves().copy()

    def is_fully_expanded(self):
        return len(self.untried_actions) == 0

    def best_child(self, c_puct=C_PUCT):
        """Select child with highest PUCT score."""
        best_score = -float('inf')
        best_child = None
        sqrt_parent = np.sqrt(self.visit_count + 1)  # +1 avoids zero
        for col, child in self.children.items():
            if child.visit_count == 0:
                q = 0.0
            else:
                q = child.total_value / child.visit_count
            # PUCT bonus: prior * sqrt(parent_visits) / (1 + child_visits)
            u = c_puct * self.prior.get(col, 0.0) * sqrt_parent / (1.0 + child.visit_count)
            score = q + u
            if score > best_score:
                best_score = score
                best_child = child
        return best_child

    def backpropagate(self, value):
        """Update all ancestors. `value` is from the perspective of the node being backpropped."""
        node = self
        while node is not None:
            node.visit_count += 1
            node.total_value += value
            value = -value          # flip perspective for parent
            node = node.parent


def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()


def mcts_search(root_state: ConnectFour,
                model: ConnectFiveNet,
                num_simulations: int = MCTS_SIMULATIONS,
                c_puct: float = C_PUCT,
                add_dirichlet: bool = True):
    """
    Run MCTS from `root_state` using the network for leaf evaluation.
    Returns:
        visit_probs: (10,) numpy array – visit distribution over columns
        root_value:  float – value of root state from its perspective
    """
    root = MCTSNode(root_state)

    # Evaluate root node with network to get priors
    logits, _ = network_eval(model, root_state)
    legal = root_state.legal_moves()
    logits_masked = np.full(COLS, -1e9)
    for col in legal:
        logits_masked[col] = logits[col]
    priors = softmax(logits_masked)   # (10,) array

    # Add Dirichlet noise at root (only during self‑play)
    if add_dirichlet and len(legal) > 1:
        noise = np.random.dirichlet([DIRICHLET_ALPHA] * len(legal))
        for i, col in enumerate(legal):
            priors[col] = (1 - DIRICHLET_EPS) * priors[col] + DIRICHLET_EPS * noise[i]

    for col in legal:
        root.prior[col] = priors[col]

    # MCTS loop
    for _ in range(num_simulations):
        node = root

        # --- SELECTION ---
        while not node.state.done and node.is_fully_expanded():
            node = node.best_child(c_puct)

        # --- EXPANSION & EVALUATION ---
        if node.state.done:
            # Terminal node – value from node's perspective
            if node.state.winner is None:
                leaf_value = 0.0
            else:
                # current_player is the loser (toggle happened after win)
                leaf_value = -1.0 if node.state.current_player != node.state.winner else 1.0
            # (in practice this is always -1 or 0)
        else:
            # Expand one untried action
            col = node.untried_actions.pop()
            child_state = node.state.clone()
            child_state.apply_move(col)
            child = MCTSNode(child_state, parent=node, action=col)
            node.children[col] = child

            # Evaluate child with network
            child_logits, child_value = network_eval(model, child_state)
            child_legal = child_state.legal_moves()

            # If child is terminal, use true outcome
            if child_state.done:
                if child_state.winner is None:
                    leaf_value = 0.0
                else:
                    leaf_value = -1.0 if child_state.current_player != child_state.winner else 1.0
            else:
                leaf_value = child_value
                # Store priors in child for future PUCT
                cm = np.full(COLS, -1e9)
                for c in child_legal:
                    cm[c] = child_logits[c]
                child_priors = softmax(cm)
                for c in child_legal:
                    child.prior[c] = child_priors[c]

            child.visit_count = 0           # will be incremented in backprop
            child.total_value = 0.0
            node = child

        # --- BACKPROPAGATION ---
        node.backpropagate(leaf_value)

    # Build visit distribution from root
    visits = np.zeros(COLS)
    for col, child in root.children.items():
        visits[col] = child.visit_count
    total = visits.sum()
    if total > 0:
        visit_probs = visits / total
    else:
        visit_probs = np.ones(COLS) / COLS   # fallback

    root_value = root.total_value / max(root.visit_count, 1)
    return visit_probs, root_value

In [ ]:
def play_self_play_game(model: ConnectFiveNet,
                        num_simulations: int = MCTS_SIMULATIONS,
                        temperature_thresh: int = TEMPERATURE_THRESH):
    """
    Play one game of the current model vs itself.
    Returns list of (state_tensor, policy_target, current_player) for every move.
    (Value targets are computed after the game ends.)
    """
    game = ConnectFour()
    trajectory = []   # (state_tensor, visit_distribution, current_player)

    move_idx = 0
    while not game.done:
        # Run MCTS
        visit_probs, _ = mcts_search(game, model,
                                     num_simulations=num_simulations,
                                     add_dirichlet=True)

        # Store the position
        state_tensor = board_to_tensor(game.board, game.current_player)
        trajectory.append((state_tensor, visit_probs, game.current_player))

        # Select action (with temperature)
        if move_idx < temperature_thresh:
            # Temperature = 1 → sample proportional to visit counts
            probs = visit_probs
        else:
            # Greedy → pick most visited
            probs = np.zeros_like(visit_probs)
            probs[np.argmax(visit_probs)] = 1.0

        col = np.random.choice(COLS, p=probs)
        # Fallback to legal if illegal (shouldn't happen, but safe)
        if col not in game.legal_moves():
            col = np.random.choice(game.legal_moves())
        game.apply_move(col)
        move_idx += 1

    # Compute value targets for each stored position
    outcome = game.winner   # 0 or 1 (or None for draw)
    training_data = []
    for state_tensor, policy_target, player in trajectory:
        if outcome is None:
            value_target = 0.0
        elif outcome == player:
            value_target = 1.0
        else:
            value_target = -1.0
        training_data.append((state_tensor, policy_target, value_target))

    return training_data, outcome